# KooLab 기본 사용법

이 노트북은 KooChemicalSimulation의 Python 인터페이스(koolab)의 기본 사용법을 소개합니다.

## 목차
1. 라이브러리 임포트
2. 간단한 1D 확산 문제
3. 2D 메시 생성
4. 결과 시각화
5. 데이터 저장 및 로딩

## 1. 라이브러리 임포트

In [ ]:
import sys
import os

# KooLab 경로 추가
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'build'))

import _core as koo
import numpy as np
import matplotlib.pyplot as plt

# 플롯 스타일 설정
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f"KooLab Version: {koo.version()}")

## 2. 로거 초기화

In [ ]:
# 로거 초기화 및 설정
koo.Logger.initialize("KooLab_Tutorial")
koo.Logger.set_level(koo.LogLevel.INFO)

koo.Logger.info("Logger initialized successfully!")
koo.Logger.debug("This is a debug message")

## 3. 간단한 2D 메시 생성

직사각형 영역에 구조화된 메시를 생성합니다.

In [ ]:
# 10x10 직사각형 메시 생성 (0,0) ~ (1,1)
mesh = koo.create_rectangular_mesh(
    x0=0.0, y0=0.0,
    x1=1.0, y1=1.0,
    nx=10, ny=10
)

print(f"Mesh created: {mesh}")
print(f"Number of nodes: {mesh.get_num_nodes()}")
print(f"Number of elements: {mesh.get_num_elements()}")

koo.Logger.info(f"Created mesh with {mesh.get_num_nodes()} nodes")

## 4. 메시 시각화

In [ ]:
# 노드 좌표 추출
num_nodes = mesh.get_num_nodes()
x_coords = []
y_coords = []

for i in range(num_nodes):
    node = mesh.get_node(i)
    if node is not None:
        x_coords.append(node.x)
        y_coords.append(node.y)

# 플롯
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(x_coords, y_coords, c='blue', s=50, alpha=0.6, edgecolors='black')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('2D Structured Mesh')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print(f"Plotted {len(x_coords)} nodes")

## 5. Element 타입 확인

In [ ]:
# 첫 번째 요소 확인
elem = mesh.get_element(0)
if elem is not None:
    print(f"Element: {elem}")
    print(f"Element type: {elem.type}")
    print(f"Number of nodes: {elem.get_num_nodes()}")
    print(f"Node IDs: {elem.get_node_ids()}")

# Element 타입 enum 확인
print("\nAvailable element types:")
print(f"  VERTEX: {koo.ElementType.VERTEX}")
print(f"  LINE: {koo.ElementType.LINE}")
print(f"  TRIANGLE: {koo.ElementType.TRIANGLE}")
print(f"  QUADRILATERAL: {koo.ElementType.QUADRILATERAL}")
print(f"  TETRAHEDRON: {koo.ElementType.TETRAHEDRON}")
print(f"  HEXAHEDRON: {koo.ElementType.HEXAHEDRON}")

## 6. 수동으로 메시 생성

In [ ]:
# 새로운 빈 메시 생성
custom_mesh = koo.MeshData()

# 4개 노드 추가 (사각형)
node1 = koo.Node(0, 0.0, 0.0, 0.0)
node2 = koo.Node(1, 1.0, 0.0, 0.0)
node3 = koo.Node(2, 1.0, 1.0, 0.0)
node4 = koo.Node(3, 0.0, 1.0, 0.0)

custom_mesh.add_node(node1)
custom_mesh.add_node(node2)
custom_mesh.add_node(node3)
custom_mesh.add_node(node4)

# 사각형 요소 추가
quad_element = koo.Element(0, koo.ElementType.QUADRILATERAL, [0, 1, 2, 3])
custom_mesh.add_element(quad_element)

print(f"Custom mesh: {custom_mesh}")

# 노드 정보 출력
for i in range(4):
    node = custom_mesh.get_node(i)
    if node:
        print(f"  {node}")

## 7. 1D 확산 시뮬레이션 데이터 생성

Python에서 간단한 1D 확산을 NumPy로 시뮬레이션합니다.

In [ ]:
# 1D 확산 파라미터
nx = 100
L = 1.0
dx = L / (nx - 1)
D = 0.01  # 확산 계수
dt = 0.001
num_steps = 1000

# 초기 조건: 가우시안 분포
x = np.linspace(0, L, nx)
C = np.exp(-((x - 0.5)**2) / (2 * 0.05**2))

# 확산 계수
alpha = D * dt / (dx**2)
print(f"CFL parameter alpha = {alpha:.4f} (should be < 0.5 for stability)")

# 시간 진화 (명시적 오일러)
C_history = [C.copy()]
times = [0]

for step in range(num_steps):
    C_new = C.copy()
    for i in range(1, nx-1):
        C_new[i] = C[i] + alpha * (C[i+1] - 2*C[i] + C[i-1])
    C = C_new
    
    # 일부 시간 스텝 저장
    if step % 100 == 0:
        C_history.append(C.copy())
        times.append((step + 1) * dt)

koo.Logger.info(f"Completed {num_steps} diffusion steps")
print(f"Saved {len(C_history)} snapshots")

## 8. 확산 결과 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# 여러 시간 스텝 플롯
for i, (C_snap, t) in enumerate(zip(C_history, times)):
    alpha_val = 0.3 + 0.7 * (i / len(C_history))
    ax.plot(x, C_snap, label=f't = {t:.3f}s', alpha=alpha_val, linewidth=2)

ax.set_xlabel('Position (m)', fontsize=12)
ax.set_ylabel('Concentration', fontsize=12)
ax.set_title('1D Diffusion Simulation', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

koo.Logger.info("Diffusion visualization complete")

## 9. 2D 히트맵 시각화

In [ ]:
# 2D 확산 데이터 생성 (간단한 예제)
nx, ny = 50, 50
x_2d = np.linspace(0, 1, nx)
y_2d = np.linspace(0, 1, ny)
X, Y = np.meshgrid(x_2d, y_2d)

# 가우시안 분포
C_2d = np.exp(-((X - 0.5)**2 + (Y - 0.5)**2) / (2 * 0.1**2))

# 히트맵 플롯
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 컨투어 플롯
contour = ax1.contourf(X, Y, C_2d, levels=20, cmap='viridis')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('2D Concentration (Contour)')
plt.colorbar(contour, ax=ax1)

# 이미지 플롯
im = ax2.imshow(C_2d, extent=[0, 1, 0, 1], origin='lower', cmap='hot')
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_title('2D Concentration (Heatmap)')
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.show()

## 10. 요약 및 다음 단계

이 노트북에서 배운 내용:
- ✅ KooLab 라이브러리 임포트 및 버전 확인
- ✅ Logger 초기화 및 사용
- ✅ 2D 메시 생성 및 시각화
- ✅ 수동 메시 생성
- ✅ 1D 확산 시뮬레이션
- ✅ 결과 시각화 (1D, 2D)

### 다음 단계:
- `02_reaction_diffusion.ipynb`: 반응-확산 시스템
- `03_real_time_viz.ipynb`: 실시간 시각화
- `04_gpu_acceleration.ipynb`: GPU 가속

In [ ]:
# 세션 정보
print("=" * 50)
print("Session Summary")
print("=" * 50)
print(f"KooLab Version: {koo.version()}")
print(f"NumPy Version: {np.__version__}")
print(f"Matplotlib Version: {plt.matplotlib.__version__}")
print("=" * 50)
koo.Logger.info("Tutorial completed successfully!")